# DS2OS: prueba de dependencia de la identidad (split por dispositivo)

In [1]:
import warnings, numpy as np, pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score, recall_score, roc_auc_score
warnings.filterwarnings("ignore"); SEED = 42

DS2 = (Path.home() / ".cache" / "kagglehub" / "datasets" / "libamariyam" / "ds2os-dataset" / "versions" / "1" / "DS2OS.csv")
SAMPLE_N = None    # pon p.ej. 120000 para ir mas rapido; None = dataset completo

df = pd.read_csv(DS2)
df["accessedNodeType"] = df["accessedNodeType"].fillna("Malicious")
df["value"] = pd.to_numeric(df["value"].replace({"False":0,"True":1,"Twenty":20,"none":0}),
                            errors="coerce").fillna(0)
CAT = ["sourceID","sourceAddress","sourceType","sourceLocation","destinationServiceAddress",
       "destinationServiceType","destinationLocation","accessedNodeAddress","accessedNodeType","operation"]
for c in CAT:
    df[c] = LabelEncoder().fit_transform(df[c].astype(str))
feats = CAT + ["value"]
y = (df["normality"].str.lower() != "normal").astype(int).values
groups = df["sourceID"].values                      # dispositivo (para el split por grupos)

if SAMPLE_N and len(y) > SAMPLE_N:
    rng = np.random.RandomState(SEED)
    idx = np.hstack([rng.choice(np.where(y==c)[0], int(round(SAMPLE_N*(y==c).mean())), replace=False)
                     for c in np.unique(y)])
    df, y, groups = df.iloc[idx].reset_index(drop=True), y[idx], groups[idx]

IDENTIDAD = ["sourceID","sourceAddress","accessedNodeAddress","destinationServiceAddress"]
feats_noid = [f for f in feats if f not in IDENTIDAD]
print(f"n={len(y):,}  ataques={100*y.mean():.2f}%  dispositivos(sourceID)={len(np.unique(groups))}")

n=357,952  ataques=2.80%  dispositivos(sourceID)=84


In [2]:
def evaluar(cols, splitter, groups=None):
    X = df[cols].values
    f1s, recs, aucs = [], [], []
    for tr, te in splitter.split(X, y, groups):
        clf = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                     random_state=SEED, n_jobs=-1)
        clf.fit(X[tr], y[tr])
        pred  = clf.predict(X[te])
        proba = clf.predict_proba(X[te])[:, 1]
        f1s.append(f1_score(y[te], pred, average="macro"))
        recs.append(recall_score(y[te], pred))            # recall de la clase ataque
        aucs.append(roc_auc_score(y[te], proba))
    return np.mean(f1s), np.mean(recs), np.mean(aucs)

skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
gkf = GroupKFold(5)

r1 = evaluar(feats,      skf)                 # CON sourceID, aleatorio
r2 = evaluar(feats,      gkf, groups)         # CON sourceID, por dispositivo
r3 = evaluar(feats_noid, gkf, groups)         # SIN identidad, por dispositivo

print(f"{'escenario':45s}{'F1-macro':>10}{'recall_ataque':>15}{'AUC':>8}")
print("-"*78)
for nombre, r in [("CON sourceID  | split aleatorio", r1),
                  ("CON sourceID  | split por dispositivo", r2),
                  ("SIN identidad | split por dispositivo", r3)]:
    print(f"{nombre:45s}{r[0]:>10.3f}{r[1]:>15.3f}{r[2]:>8.3f}")

escenario                                      F1-macro  recall_ataque     AUC
------------------------------------------------------------------------------
CON sourceID  | split aleatorio                   0.947          1.000   0.999
CON sourceID  | split por dispositivo             0.694          0.272   0.930
SIN identidad | split por dispositivo             0.700          0.416   0.958


## Lectura